# GBD Algoritmus – Vizualizácia krokov

Tento notebook vizualizuje každý krok Graph-Based Decomposition (GBD) algoritmu:

1. **Vstupný obrázok** – binárna mriežka
2. **Konkávne vrcholy** – detekcia vnútorných rohov
3. **Príslušné pixely** – pixel každého konkávneho vrcholu
4. **Všetky tetivy** – všetky kandidátske spoje medzi vrcholmi
5. **Vybrané tetivy (MIS Level 1)** – optimálna nezávislá množina tetív
6. **Zostatok osamotených vrcholov** – vrcholy neriešené v Level 1
7. **Level 2 rezy** – lúče ku najbližšej hrane/tetive
8. **Konečný výsledok** – obdĺžniková dekompozícia

In [380]:
import sys
import os
sys.path.insert(0, os.path.abspath('../../..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from collections import defaultdict, deque

## Algoritmus – kópia a úprava z `graph_based.py`

Funkcie sú skopírované sem aby bolo možné ich upravovať bez zmeny zdrojového súboru.

In [381]:
def find_concave_corners(grid):
    """Nájde konkávne rohy (vnútorné rohy) v binárnej mriežke.

    Konkávny roh je aktívny pixel (1) ktorého dvaja ortogonálni susedia
    sú tiež aktívni, ale diagonálny sused medzi nimi je prázdny (0).

    Returns:
        list[tuple]: (id, px, py, corner_label, cx, cy)
    """
    rows, cols = grid.shape
    concave_points = []
    idx = 0

    for y in range(rows):
        for x in range(cols):
            if grid[y, x] == 1:
                def g(dy, dx):
                    ny, nx_ = y + dy, x + dx
                    if 0 <= ny < rows and 0 <= nx_ < cols:
                        return grid[ny, nx_]
                    return 0

                t, b = g(-1, 0), g(1, 0)
                l, r = g(0, -1), g(0, 1)
                tl, tr = g(-1, -1), g(-1, 1)
                bl, br = g(1, -1), g(1, 1)

                if t == 1 and l == 1 and tl == 0:
                    concave_points.append((idx, x, y, ["top-left"],  x - 0.5, y - 0.5))
                    idx += 1
                if t == 1 and r == 1 and tr == 0:
                    concave_points.append((idx, x, y, ["top-right"], x + 0.5, y - 0.5))
                    idx += 1
                if b == 1 and r == 1 and br == 0:
                    concave_points.append((idx, x, y, ["bottom-right"], x + 0.5, y + 0.5))
                    idx += 1
                if b == 1 and l == 1 and bl == 0:
                    concave_points.append((idx, x, y, ["bottom-left"],  x - 0.5, y + 0.5))
                    idx += 1

    return concave_points

In [382]:
def find_cogrid_pairs(concave_vertices, grid):
    """Nájde všetky platné tetivy (chords) medzi konkávnymi vrcholmi."""
    rows, cols = grid.shape
    chords = []
    chord_id = 0

    enriched = []
    for vid, px, py, corner, cx, cy in concave_vertices:
        ctype = corner[0] if isinstance(corner, list) else corner
        enriched.append({"id": vid, "px": px, "py": py, "corner": ctype, "cx": cx, "cy": cy})

    # Horizontálne tetivy – skupiny s rovnakým cy
    y_groups = defaultdict(list)
    for v in enriched:
        y_groups[v["cy"]].append(v)

    for cy, verts in y_groups.items():
        sorted_verts = sorted(verts, key=lambda v: v["cx"])
        for i in range(len(sorted_verts) - 1):
            v1, v2 = sorted_verts[i], sorted_verts[i + 1]
            x_start = int(min(v1["px"], v2["px"]))
            x_end   = int(max(v1["px"], v2["px"]))
            check_y_above = int(cy - 0.5)
            check_y_below = int(cy + 0.5)
            path_above = (0 <= check_y_above < rows and
                          all(grid[check_y_above, x] == 1 for x in range(x_start, x_end)))
            path_below = (0 <= check_y_below < rows and
                          all(grid[check_y_below, x] == 1 for x in range(x_start, x_end)))
            if path_above and path_below:
                chords.append({
                    "id": chord_id, "type": "horizontal",
                    "v1_id": v1["id"], "v2_id": v2["id"],
                    "v1": (v1["px"], v1["py"]), "v2": (v2["px"], v2["py"]),
                    "y_line": cy, "x_range": (v1["cx"], v2["cx"]),
                    "interior_side": "both"
                })
                chord_id += 1

    # Vertikálne tetivy – skupiny s rovnakým cx
    x_groups = defaultdict(list)
    for v in enriched:
        x_groups[v["cx"]].append(v)

    for cx, verts in x_groups.items():
        sorted_verts = sorted(verts, key=lambda v: v["cy"])
        for i in range(len(sorted_verts) - 1):
            v1, v2 = sorted_verts[i], sorted_verts[i + 1]
            y_start = int(min(v1["py"], v2["py"]))
            y_end   = int(max(v1["py"], v2["py"]))
            check_x_left  = int(cx - 0.5)
            check_x_right = int(cx + 0.5)
            path_left  = (0 <= check_x_left  < cols and
                          all(grid[y, check_x_left]  == 1 for y in range(y_start, y_end)))
            path_right = (0 <= check_x_right < cols and
                          all(grid[y, check_x_right] == 1 for y in range(y_start, y_end)))
            if path_left and path_right:
                chords.append({
                    "id": chord_id, "type": "vertical",
                    "v1_id": v1["id"], "v2_id": v2["id"],
                    "v1": (v1["px"], v1["py"]), "v2": (v2["px"], v2["py"]),
                    "x_line": cx, "y_range": (v1["cy"], v2["cy"]),
                    "interior_side": "both"
                })
                chord_id += 1

    return chords

In [383]:
def chords_intersect(c1, c2):
    """Zistí či sa dve tetivy (horizontálna a vertikálna) pretínajú."""
    if c1['type'] == c2['type']:
        return False
    h = c1 if c1['type'] == 'horizontal' else c2
    v = c2 if c1['type'] == 'horizontal' else c1
    h_y = h.get('y_line') if h.get('y_line') is not None else h.get('y')
    v_x = v.get('x_line') if v.get('x_line') is not None else v.get('x')
    h_x_min, h_x_max = h['x_range']
    v_y_min, v_y_max = v['y_range']
    return (h_x_min <= v_x <= h_x_max) and (v_y_min <= h_y <= v_y_max)


def gbd_algorithm_level1(concave_vertices, grid, verbose=False):
    """Level 1: MIS cez Maximum Bipartite Matching (König's theorem).

    Returns:
        (all_candidate_chords, level1_chords, remaining_vertices)
    """
    all_candidate_chords = find_cogrid_pairs(concave_vertices, grid)

    if not all_candidate_chords:
        return all_candidate_chords, [], concave_vertices

    h_chords = [c for c in all_candidate_chords if c['type'] == 'horizontal']
    v_chords = [c for c in all_candidate_chords if c['type'] == 'vertical']

    B = nx.Graph()
    h_nodes = [f"h{c['id']}" for c in h_chords]
    v_nodes = [f"v{c['id']}" for c in v_chords]
    B.add_nodes_from(h_nodes, bipartite=0)
    B.add_nodes_from(v_nodes, bipartite=1)

    for h in h_chords:
        for v in v_chords:
            if chords_intersect(h, v):
                B.add_edge(f"h{h['id']}", f"v{v['id']}")

    matching = nx.bipartite.maximum_matching(B, top_nodes=h_nodes)
    mvc = nx.bipartite.to_vertex_cover(B, matching, top_nodes=h_nodes)

    selected_ids = []
    for c in all_candidate_chords:
        node_id = f"{'h' if c['type'] == 'horizontal' else 'v'}{c['id']}"
        if node_id not in mvc:
            selected_ids.append(c['id'])

    level1_chords = [c for c in all_candidate_chords if c['id'] in selected_ids]

    used_v_ids = set()
    for c in level1_chords:
        used_v_ids.add(c['v1_id'])
        used_v_ids.add(c['v2_id'])
    remaining_vertices = [v for v in concave_vertices if v[0] not in used_v_ids]

    if verbose:
        print(f"Kandidátske tetivy: {len(all_candidate_chords)}")
        print(f"MIS vybrané (Level 1): {len(level1_chords)}")
        print(f"Zostatok pre Level 2: {len(remaining_vertices)}")

    return all_candidate_chords, level1_chords, remaining_vertices

In [384]:
def gbd_algorithm_level2(remaining_vertices, grid, level1_chords, verbose=False):
    """Level 2: Lúčové vystreľovanie ku najbližšej hrane alebo existujúcej tetive."""
    level2_chords = []

    def get_allowed_directions(corners):
        directions = set()
        c = corners[0] if isinstance(corners, list) else corners
        if "top"    in c: directions.add("down")
        if "bottom" in c: directions.add("up")
        if "left"   in c: directions.add("right")
        if "right"  in c: directions.add("left")
        return directions

    def is_blocked_by_chord(cx, cy, direction, blocking_chords):
        for chord in blocking_chords:
            c_type = chord.get('type')
            if c_type == 'vertical' and direction in ('left', 'right'):
                c_line  = chord.get('x_line') if chord.get('x_line') is not None else chord.get('x')
                c_range = chord.get('y_range')
                if c_line is not None and abs(cx - c_line) < 0.01:
                    if c_range and c_range[0] <= cy <= c_range[1]:
                        return True
            elif c_type == 'horizontal' and direction in ('up', 'down'):
                c_line  = chord.get('y_line') if chord.get('y_line') is not None else chord.get('y')
                c_range = chord.get('x_range')
                if c_line is not None and abs(cy - c_line) < 0.01:
                    if c_range and c_range[0] <= cx <= c_range[1]:
                        return True
        return False

    def extend_chord(v_cx, v_cy, direction, grid, level2_chords, level1_cuts):
        blocking_all = level1_cuts + level2_chords
        curr_cx, curr_cy = v_cx, v_cy
        path = [(curr_cx, curr_cy)]
        rows, cols = grid.shape
        max_steps = (rows + cols) * 2
        steps = 0
        while steps < max_steps:
            steps += 1
            if   direction == 'right': curr_cx += 0.5
            elif direction == 'left':  curr_cx -= 0.5
            elif direction == 'down':  curr_cy += 0.5
            elif direction == 'up':    curr_cy -= 0.5
            if curr_cx < 0 or curr_cx > cols or curr_cy < 0 or curr_cy > rows:
                return path
            check_x = int(curr_cx - 0.5) if direction == 'left'  else int(curr_cx)
            check_y = int(curr_cy - 0.5) if direction == 'up'    else int(curr_cy)
            if 0 <= check_x < cols and 0 <= check_y < rows:
                if grid[check_y, check_x] == 0:
                    return path
            else:
                return path
            if is_blocked_by_chord(curr_cx, curr_cy, direction, blocking_all):
                path.append((curr_cx, curr_cy))
                return path
            if curr_cx % 1.0 == 0 or curr_cy % 1.0 == 0:
                if (curr_cx, curr_cy) not in path:
                    path.append((curr_cx, curr_cy))
        return path

    def build_chord(vertex, direction, path):
        if len(path) < 2:
            return None
        idx, px, py, corner, v_cx, v_cy = vertex
        end_cx, end_cy = path[-1]
        ctype = corner[0] if isinstance(corner, list) else corner
        if direction in ('left', 'right'):
            return {
                'type': 'horizontal', 'v1': (v_cx, v_cy), 'v2': (end_cx, end_cy),
                'y_line': v_cy, 'x_range': (min(v_cx, end_cx), max(v_cx, end_cx)),
                'interior_side': "below" if "top" in ctype else "above",
                'direction': direction, 'v1_id': idx, 'v2_id': None
            }
        else:
            return {
                'type': 'vertical', 'v1': (v_cx, v_cy), 'v2': (end_cx, end_cy),
                'x_line': v_cx, 'y_range': (min(v_cy, end_cy), max(v_cy, end_cy)),
                'interior_side': "right" if "left" in ctype else "left",
                'direction': direction, 'v1_id': idx, 'v2_id': None
            }

    for vertex in remaining_vertices:
        idx, x, y, corners, v_cx, v_cy = vertex
        allowed_directions = get_allowed_directions(corners)
        candidates = []
        for direction in allowed_directions:
            pixels = extend_chord(v_cx, v_cy, direction, grid, level2_chords, level1_chords)
            chord = build_chord(vertex, direction, pixels)
            if chord:
                chord['length'] = len(pixels)
                candidates.append(chord)
        if candidates:
            selected = min(candidates, key=lambda c: c['length'])
            level2_chords.append(selected)

    return level2_chords

In [385]:
def convert_chords_to_precise_cuts(chords):
    """Normalizuje tetivy na formát presných rezov."""
    precise_cuts = []
    for chord in chords:
        if chord['type'] == 'horizontal':
            if 'y_line' in chord:
                precise_cuts.append({
                    'type': 'horizontal',
                    'y_line': chord['y_line'],
                    'x_range': chord['x_range'],
                    'side': chord.get('interior_side') or chord.get('direction') or chord.get('side')
                })
            else:
                interior = chord.get('interior_side') or chord.get('direction')
                y_line = chord['y'] - 0.5 if interior == 'below' else chord['y'] + 0.5
                precise_cuts.append({'type': 'horizontal', 'y_line': y_line,
                                     'x_range': chord['x_range'], 'side': interior})
        else:
            if 'x_line' in chord:
                precise_cuts.append({
                    'type': 'vertical',
                    'x_line': chord['x_line'],
                    'y_range': chord['y_range'],
                    'side': chord.get('interior_side') or chord.get('direction') or chord.get('side')
                })
            else:
                interior = chord.get('interior_side') or chord.get('direction')
                x_line = chord['x'] - 0.5 if interior == 'right' else chord['x'] + 0.5
                precise_cuts.append({'type': 'vertical', 'x_line': x_line,
                                     'y_range': chord['y_range'], 'side': interior})
    return precise_cuts


def find_rectangles_from_cuts(grid, cuts, verbose=False):
    """Dekomponuje binárnu mriežku na obdĺžniky pomocou flood fill."""
    rows, cols = grid.shape
    region_map = np.full((rows, cols), -1, dtype=int)
    region_id  = 0

    def is_cut_between(y1, x1, y2, x2):
        for cut in cuts:
            if cut['type'] == 'vertical':
                x_line = cut['x_line']
                y_start, y_end = cut['y_range']
                if y1 == y2 and abs(x1 - x2) == 1:
                    x_border = (min(x1, x2) + max(x1, x2)) / 2.0
                    if abs(x_border - x_line) < 0.1 and y_start < y1 < y_end:
                        return True
            else:
                y_line = cut['y_line']
                x_start, x_end = cut['x_range']
                if x1 == x2 and abs(y1 - y2) == 1:
                    y_border = (min(y1, y2) + max(y1, y2)) / 2.0
                    if abs(y_border - y_line) < 0.1 and x_start < x1 < x_end:
                        return True
        return False

    def flood_fill(start_y, start_x, rid):
        queue = deque([(start_y, start_x)])
        region_map[start_y, start_x] = rid
        points = []
        while queue:
            y, x = queue.popleft()
            points.append((x, y))
            for dy, dx in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                ny, nx_ = y + dy, x + dx
                if (0 <= ny < rows and 0 <= nx_ < cols and
                        grid[ny, nx_] == 1 and region_map[ny, nx_] == -1 and
                        not is_cut_between(y, x, ny, nx_)):
                    region_map[ny, nx_] = rid
                    queue.append((ny, nx_))
        return points

    def largest_rect_in_region(points):
        if not points:
            return []
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        W, H = x_max - x_min + 1, y_max - y_min + 1
        local = np.zeros((H, W), dtype=int)
        for (x, y) in points:
            local[y - y_min, x - x_min] = 1
        rectangles = []
        remaining = local.copy()

        def maximal_rect_histogram(m):
            rc, cc = m.shape
            height = np.zeros(cc + 1, dtype=int)
            best_area, best_rect = 0, None
            for r in range(rc):
                for c in range(cc):
                    height[c] = height[c] + 1 if m[r, c] else 0
                stack = [-1]
                for c in range(cc + 1):
                    while height[c] < height[stack[-1]]:
                        h = height[stack.pop()]
                        w = c - stack[-1] - 1
                        area = h * w
                        if area > best_area:
                            best_area = area
                            best_rect = (r - h + 1, stack[-1] + 1, r, c - 1)
                    stack.append(c)
            return best_area, best_rect

        while remaining.sum() > 0:
            area, rect = maximal_rect_histogram(remaining)
            if rect is None or area == 0:
                break
            r1, c1, r2, c2 = rect
            ax_, ay_ = c1 + x_min, r1 + y_min
            w_r, h_r = c2 - c1 + 1, r2 - r1 + 1
            rectangles.append({'min_x': ax_, 'min_y': ay_,
                                'max_x': ax_ + w_r - 1, 'max_y': ay_ + h_r - 1,
                                'width': w_r, 'height': h_r, 'area': w_r * h_r})
            remaining[r1:r2 + 1, c1:c2 + 1] = 0

        return rectangles

    all_rectangles = []
    for i in range(rows):
        for j in range(cols):
            if grid[i, j] == 1 and region_map[i, j] == -1:
                pts = flood_fill(i, j, region_id)
                region_id += 1
                all_rectangles.extend(largest_rect_in_region(pts))
    return all_rectangles

In [386]:
def gbd_algorithm_complete(grid, verbose=False):
    """Celý GBD algoritmus – vracia aj medzivýsledky pre vizualizáciu.

    Returns:
        dict s kľúčmi:
            concave_vertices, all_candidate_chords,
            level1_chords, remaining_vertices,
            level2_chords, rectangles, coverage
    """
    # Krok 1: detekcia konkávnych vrcholov
    concave_vertices = find_concave_corners(grid)

    # Krok 2+3: Level 1 – MIS tetivy
    all_candidate_chords, level1_chords, remaining_vertices = gbd_algorithm_level1(
        concave_vertices, grid, verbose
    )

    # Krok 4: Level 2 – lúčové vystreľovanie
    level2_chords = gbd_algorithm_level2(
        remaining_vertices, grid, level1_chords, verbose
    )

    # Krok 5: rezy → obdĺžniky
    all_chords  = level1_chords + level2_chords
    precise_cuts = convert_chords_to_precise_cuts(all_chords)
    rectangles   = find_rectangles_from_cuts(grid, precise_cuts, verbose)

    # Overenie pokrytia
    orig_px = set(zip(*np.where(grid == 1)))
    cov_px  = set()
    for r in rectangles:
        for y in range(r['min_y'], r['max_y'] + 1):
            for x in range(r['min_x'], r['max_x'] + 1):
                cov_px.add((y, x))
    is_covered = (cov_px == orig_px)

    if verbose:
        print(f"Konkávne vrcholy: {len(concave_vertices)}")
        print(f"Kandidáti: {len(all_candidate_chords)}, Level1: {len(level1_chords)}, Level2: {len(level2_chords)}")
        print(f"Obdĺžnikov: {len(rectangles)}, Pokrytie: {'OK' if is_covered else 'CHYBA'}")

    return {
        'concave_vertices':    concave_vertices,
        'all_candidate_chords': all_candidate_chords,
        'level1_chords':        level1_chords,
        'remaining_vertices':   remaining_vertices,
        'level2_chords':        level2_chords,
        'rectangles':           rectangles,
        'coverage':             is_covered,
    }

## Vizualizačná funkcia

In [387]:
CORNER_COLORS = {
    'top-left':     '#FF6B6B',
    'top-right':    '#4ECDC4',
    'bottom-right': '#F7DC6F',
    'bottom-left':  '#BB8FCE',
}

RECT_PALETTE = [
    '#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8',
    '#F7DC6F', '#BB8FCE', '#85C1E2', '#F8B88B', '#A8E6CF',
    '#FF5733', '#33FF57', '#3357FF', '#FF33F5', '#F5FF33',
]


def _make_rgb(grid):
    """Biele pozadie (0), cierny objekt (1)."""
    img = np.full((*grid.shape, 3), 255, dtype=np.uint8)
    img[grid == 1] = [0, 0, 0]
    return img


def _draw_chord(ax, chord, color, lw=2.0, ls='-', alpha=1.0, zorder=3):
    if chord['type'] == 'horizontal':
        y = chord['y_line']
        x0, x1 = chord['x_range']
        ax.plot([x0, x1], [y, y], color=color, lw=lw, ls=ls,
                alpha=alpha, solid_capstyle='round', zorder=zorder)
    else:
        x = chord['x_line']
        y0, y1 = chord['y_range']
        ax.plot([x, x], [y0, y1], color=color, lw=lw, ls=ls,
                alpha=alpha, solid_capstyle='round', zorder=zorder)


def _draw_l2_chord(ax, chord, color, lw=4.5, alpha=1.0, zorder=4):
    """Nakresli Level-2 rez od vrchola v smere rezu."""
    direction = chord.get('direction', '')
    vx, vy = chord['v1']
    if chord['type'] == 'horizontal':
        y = chord['y_line']
        x0, x1 = chord['x_range']
        if direction == 'right':
            x0 = vx
        elif direction == 'left':
            x1 = vx
        ax.plot([x0, x1], [y, y], color=color, lw=lw,
                alpha=alpha, solid_capstyle='round', zorder=zorder)
    else:
        x = chord['x_line']
        y0, y1 = chord['y_range']
        if direction == 'down':
            y0 = vy
        elif direction == 'up':
            y1 = vy
        ax.plot([x, x], [y0, y1], color=color, lw=lw,
                alpha=alpha, solid_capstyle='round', zorder=zorder)
    ax.plot(vx, vy, 'o', color=color, markersize=7, zorder=zorder+1,
            markeredgecolor='white', markeredgewidth=1.0)

def _setup_ax(ax, grid, title):
    ax.imshow(_make_rgb(grid), origin='upper', interpolation='nearest')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=8)
    ax.axis('off')


def _get_intersection_point(h_chord, v_chord):
    return v_chord['x_line'], h_chord['y_line']


def _new_fig(grid, cell_size):
    rows, cols = grid.shape
    w = max(cols * cell_size / 96, 4)
    h = max(rows * cell_size / 96, 4)
    fig, ax = plt.subplots(figsize=(w, h))
    return fig, ax


def _save_and_show(fig, save_dir, name, show):
    if save_dir:
        import os
        os.makedirs(save_dir, exist_ok=True)
        path = os.path.join(save_dir, f"{name}.png")
        fig.savefig(path, bbox_inches='tight', dpi=150)
        print(f"Ulozene: {path}")
    if show:
        plt.show()
    plt.close(fig)


def visualize_gbd_steps(result, grid, cell_size=80, save_dir=None, show=True):
    """Vizualizacia kazdeho kroku GBD algoritmu - kazdy krok samostatny obrazok."""
    concave_vertices     = result['concave_vertices']
    all_candidate_chords = result['all_candidate_chords']
    level1_chords        = result['level1_chords']
    remaining_vertices   = result['remaining_vertices']
    level2_chords        = result['level2_chords']
    rectangles           = result['rectangles']

    l1_ids     = {c['id'] for c in level1_chords}
    used_v_ids = set()
    for c in level1_chords:
        used_v_ids.add(c['v1_id'])
        used_v_ids.add(c['v2_id'])
    remaining_ids = {v[0] for v in remaining_vertices}
    h_cands = [c for c in all_candidate_chords if c['type'] == 'horizontal']
    v_cands = [c for c in all_candidate_chords if c['type'] == 'vertical']

    # 1: Vstupny obrazok
    fig, ax = _new_fig(grid, cell_size)
    _setup_ax(ax, grid,
              f"1. Vstupny obrazok  ({grid.shape[0]}x{grid.shape[1]}, {int(grid.sum())} px)")
    _save_and_show(fig, save_dir, '01_vstupny_obrazok', show)

    # 2: Konkavne vrcholy
    fig, ax = _new_fig(grid, cell_size)
    _setup_ax(ax, grid, f"2. Konkavne vrcholy ({len(concave_vertices)})")
    for idx, px, py, corner, cx, cy in concave_vertices:
        ctype = corner[0] if isinstance(corner, list) else corner
        c = CORNER_COLORS.get(ctype, 'red')
        ax.plot(cx, cy, 'o', color=c, markersize=25,
                markeredgecolor='white', markeredgewidth=1.2, zorder=5)
        ax.text(cx, cy, str(idx), color='black', fontsize=20,
                ha='center', va='center', fontweight='bold', zorder=6)
    ax.legend(handles=[mpatches.Patch(color=v, label=k)
                        for k, v in CORNER_COLORS.items()],
              fontsize=9, loc='lower right', framealpha=0.8, handlelength=1.2)
    _save_and_show(fig, save_dir, '02_konkavne_vrcholy', show)

    # 3: Prislusne pixely
    fig, ax = _new_fig(grid, cell_size)
    _setup_ax(ax, grid, '3. Prislusne pixely')
    cmap = plt.cm.tab20
    n = max(len(concave_vertices), 1)
    for i, (idx, px, py, corner, cx, cy) in enumerate(concave_vertices):
        color = cmap(i / n)
        ax.add_patch(plt.Rectangle((px - 0.5, py - 0.5), 1, 1,
                                   facecolor=color, alpha=0.88,
                                   edgecolor='white', lw=0.5, zorder=3))
        ax.text(px, py, str(idx), color='white', fontsize=20,
                ha='center', va='center', fontweight='bold', zorder=6)
    _save_and_show(fig, save_dir, '03_prislusne_pixely', show)

    # 4: Vsetky tetivy + krize priesecnikov
    fig, ax = _new_fig(grid, cell_size)
    _setup_ax(ax, grid, f"4. Vsetky tetivy ({len(all_candidate_chords)}) + priesecniky")
    for chord in all_candidate_chords:
        color = '#45B7D1' if chord['type'] == 'horizontal' else '#FFA07A'
        _draw_chord(ax, chord, color=color, lw=6.0, alpha=0.95)
    for h in h_cands:
        for v in v_cands:
            if chords_intersect(h, v):
                ix, iy = _get_intersection_point(h, v)
                ax.plot(ix, iy, 'x', color='red', markersize=25,
                        markeredgewidth=3.5, zorder=6)
    for idx, px, py, corner, cx, cy in concave_vertices:
        ax.plot(cx, cy, 'yo', markersize=25, zorder=5)
    ax.legend(handles=[
        mpatches.Patch(color='#45B7D1', label='horizontalne'),
        mpatches.Patch(color='#FFA07A', label='vertikalne'),
        plt.Line2D([0], [0], marker='x', color='white', lw=0,
                   markersize=25, markeredgewidth=2.5, label='priesecnik'),
    ], fontsize=9, loc='lower right', framealpha=0.8)
    _save_and_show(fig, save_dir, '04_vsetky_tetivy', show)

    # 5: Bipartitny graf + MIS
    h_chords_all = [c for c in all_candidate_chords if c['type'] == 'horizontal']
    v_chords_all = [c for c in all_candidate_chords if c['type'] == 'vertical']
    n_max = max(len(h_chords_all), len(v_chords_all), 1)

    fig_w = max(7, n_max * 1.1)
    fig_h = max(5, n_max * 1.0 + 2.5)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-1.5, n_max + 1.5)
    ax.invert_yaxis()
    ax.axis('off')
    ax.set_title('5. Bipartitny graf + MIS  (zelene = MIS)',
                 fontsize=13, fontweight='bold', pad=10, color='black')

    h_pos = {c['id']: (0.0, i) for i, c in enumerate(h_chords_all)}
    v_pos = {c['id']: (1.0, i) for i, c in enumerate(v_chords_all)}

    for h in h_chords_all:
        for v in v_chords_all:
            if chords_intersect(h, v):
                hx, hy = h_pos[h['id']]
                vx, vy = v_pos[v['id']]
                ax.plot([hx, vx], [hy, vy], color='#999999',
                        lw=1.2, alpha=0.6, zorder=1)

    for c in h_chords_all:
        x, y = h_pos[c['id']]
        color = '#2ECC71' if c['id'] in l1_ids else '#E74C3C'
        ax.plot(x, y, 'o', color=color, markersize=25, zorder=3,
                markeredgecolor='#333333', markeredgewidth=1.0)
        ax.text(x, y, f"h{c['id']}", color='white', fontsize=12,
                ha='center', va='center', fontweight='bold', zorder=4)

    for c in v_chords_all:
        x, y = v_pos[c['id']]
        color = '#2ECC71' if c['id'] in l1_ids else '#E74C3C'
        ax.plot(x, y, 's', color=color, markersize=25, zorder=3,
                markeredgecolor='#333333', markeredgewidth=1.0)
        ax.text(x, y, f"v{c['id']}", color='white', fontsize=12,
                ha='center', va='center', fontweight='bold', zorder=4)

    ax.text(0.0, -0.8, 'Horizontalne', ha='center', fontsize=12,
            color='#2980B9', fontweight='bold')
    ax.text(1.0, -0.8, 'Vertikalne', ha='center', fontsize=12,
            color='#E67E22', fontweight='bold')

    ax.legend(handles=[
        mpatches.Patch(color='#2ECC71', label='MIS (vybrane)'),
        mpatches.Patch(color='#E74C3C', label='zamietnute'),
    ], fontsize=10, loc='lower center', framealpha=0.9,
       facecolor='black', edgecolor='#CCCCCC',
       bbox_to_anchor=(0.5, 0.02))
    _save_and_show(fig, save_dir, '05_bipartitny_graf', show)

    # 6: Vybrane tetivy MIS Level 1
    fig, ax = _new_fig(grid, cell_size)
    _setup_ax(ax, grid,
              f"6. MIS Level 1  (vybrane {len(level1_chords)} / "
              f"zamietnute {len(all_candidate_chords) - len(level1_chords)})")
    for chord in all_candidate_chords:
        if chord['id'] in l1_ids:
            _draw_chord(ax, chord, color='#2ECC71', lw=6.5)
        else:
            _draw_chord(ax, chord, color='#E74C3C', lw=4.2, ls='--', alpha=0.6)
    for idx, px, py, corner, cx, cy in concave_vertices:
        ax.plot(cx, cy, 'yo', markersize=25, zorder=5)
    ax.legend(handles=[
        mpatches.Patch(color='#2ECC71', label='vybrane (MIS)'),
        mpatches.Patch(color='#E74C3C', label='zamietnute'),
    ], fontsize=9, loc='lower right', framealpha=0.8)
    _save_and_show(fig, save_dir, '06_mis_level1', show)

    # 7: Osamotene vrcholy po Level 1
    fig, ax = _new_fig(grid, cell_size)
    _setup_ax(ax, grid, f"7. Osamotene vrcholy po Level 1 ({len(remaining_vertices)})")
    for chord in level1_chords:
        _draw_chord(ax, chord, color='#2ECC71', lw=6.5, alpha=0.8)
    for idx, px, py, corner, cx, cy in concave_vertices:
        if idx in remaining_ids:
            ax.plot(cx, cy, 'o', color='#FF8C00', markersize=25,
                    markeredgecolor='red', markeredgewidth=1.8, zorder=6)
            ax.text(cx, cy, str(idx), color='white', fontsize=9,
                    ha='center', va='center', fontweight='bold', zorder=7)
        else:
            ax.plot(cx, cy, 'o', color='#2ECC71', markersize=25,
                    alpha=0.6, zorder=5)
    ax.legend(handles=[
        mpatches.Patch(color='#FF8C00', label='neriešene'),
        mpatches.Patch(color='#2ECC71', label='vyriešene L1'),
    ], fontsize=9, loc='lower right', framealpha=0.8)
    _save_and_show(fig, save_dir, '07_osamotene_vrcholy', show)

    # 8: Level 2 rezy
    # _find_l2_endpoint re-spusta ray-casting po 0.5 krokoch pouzivajuc
    # level1+level2 chords ako blokery, takze rezy su spravne dlhe a orientovane
    fig, ax = _new_fig(grid, cell_size)
    _setup_ax(ax, grid, f"8. Level 2 rezy ({len(level2_chords)})  - luc ku hrane")
    for chord in level1_chords:
        _draw_chord(ax, chord, color='#2ECC71', lw=6.5, alpha=0.8)
    for chord in level2_chords:
        _draw_l2_chord(ax, chord, '#9B59B6', lw=6.5)
    for idx, px, py, corner, cx, cy in concave_vertices:
        color = '#FF8C00' if idx in remaining_ids else '#2ECC71'
        ax.plot(cx, cy, 'o', color=color, markersize=25, alpha=0.8, zorder=5)
    ax.legend(handles=[
        mpatches.Patch(color='#2ECC71', label='Level 1'),
        mpatches.Patch(color='#9B59B6', label='Level 2'),
    ], fontsize=9, loc='lower right', framealpha=0.8)
    _save_and_show(fig, save_dir, '08_level2_rezy', show)

    # 9: Konecny vysledok
    fig, ax = _new_fig(grid, cell_size)
    ax.imshow(_make_rgb(grid), origin='upper', interpolation='nearest')
    for i, r in enumerate(rectangles):
        x, y = r['min_x'], r['min_y']
        w, h = r['width'],  r['height']
        ax.add_patch(plt.Rectangle(
            (x - 0.5, y - 0.5), w, h,
            facecolor=RECT_PALETTE[i % len(RECT_PALETTE)],
            alpha=0.72, edgecolor='white', lw=1.0, zorder=3))
    ax.set_title(f"9. Konecny vysledok ({len(rectangles)} oblznikov)",
                 fontsize=13, fontweight='bold', pad=8)
    ax.axis('off')
    _save_and_show(fig, save_dir, '09_konecny_vysledok', show)


## Testovacie obrázky

In [388]:
img_small = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 1, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 1, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 0, 1, 0, 0],
    [0, 1, 1, 1, 1, 1, 0, 1, 0, 0],
    [0, 0, 0, 0, 1, 1, 1, 1, 0, 0],
    [0, 0, 0, 0, 1, 1, 1, 1, 0, 0],
    [0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
], dtype=np.uint8)

img_cross = np.array([
    [0, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 0],
], dtype=np.uint8)

img_lshape = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 1, 0],
    [0, 1, 1, 1, 1, 1, 1, 0],
    [0, 1, 1, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
], dtype=np.uint8)

print("img_small:",  img_small.shape,  "| aktívnych px:", int(img_small.sum()))
print("img_cross:",  img_cross.shape,  "| aktívnych px:", int(img_cross.sum()))
print("img_lshape:", img_lshape.shape, "| aktívnych px:", int(img_lshape.sum()))

img_small: (11, 10) | aktívnych px: 34
img_cross: (6, 6) | aktívnych px: 20
img_lshape: (7, 8) | aktívnych px: 18


## Spustenie vizualizácie

In [389]:
# ── img_small ─────────────────────────────────────────────────────────────
# result_small = gbd_algorithm_complete(img_small, verbose=True)
# print()
# visualize_gbd_steps(result_small, img_small, cell_size=80)

In [390]:
# Debug: print Level 2 chord data
result = result_small  # change to result_cross / result_lshape as needed
print("=== Level 2 chords ===")
for ch in result['level2_chords']:
    print(
        f"type={ch['type']:10s}  dir={ch.get('direction',''):6s}"
        f"  v1={ch['v1']}  v2={ch.get('v2','?')}"
        f"  x_range={ch.get('x_range','?')}  y_range={ch.get('y_range','?')}"
        f"  y_line={ch.get('y_line','?')}  x_line={ch.get('x_line','?')}"
    )


=== Level 2 chords ===
type=horizontal  dir=left    v1=(2.5, 3.5)  v2=(2.0, 3.5)  x_range=(2.0, 2.5)  y_range=?  y_line=3.5  x_line=?
type=horizontal  dir=right   v1=(6.5, 5.5)  v2=(7.0, 5.5)  x_range=(6.5, 7.0)  y_range=?  y_line=5.5  x_line=?


In [391]:
# # ── Kríž ──────────────────────────────────────────────────────────────────
# result_cross = gbd_algorithm_complete(img_cross, verbose=True)
# print()
# visualize_gbd_steps(result_cross, img_cross, cell_size=100)

In [392]:
# # ── L-tvar ────────────────────────────────────────────────────────────────
# result_l = gbd_algorithm_complete(img_lshape, verbose=True)
# print()
# visualize_gbd_steps(result_l, img_lshape, cell_size=100)

In [ ]:
img_large = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
], dtype=np.uint8)

img_small = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0, 0, 0], # Pôvodne riadky 1-2, stĺpce 7-10 (4->2)
    [0, 0, 1, 1, 1, 0, 0, 0, 0, 0], # Pôvodne riadky 3-5
    [0, 0, 0, 1, 1, 1, 1, 1, 0, 0], # Pôvodne riadky 6-12 (šírka 10->5)
    [0, 0, 0, 1, 1, 1, 1, 1, 0, 0],
    [0, 1, 1, 1, 1, 1, 1, 0, 0, 0], # Pôvodne riadky 13-14 (šírka 13->6)
    [0, 0, 1, 1, 1, 1, 1, 1, 1, 0], # Pôvodne riadky 15-17 (šírka 15->7)
    [0, 0, 1, 1, 1, 0, 0, 0, 0, 0], # Pôvodne riadky 18-20 (šírka 7->3)
    [0, 0, 1, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 0, 0], # Pôvodne riadky 21-22 (šírka 4->2)
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
], dtype=np.uint8)

result_l = gbd_algorithm_complete(img_large, verbose=True)
print()
visualize_gbd_steps(result_l, img_large, cell_size=100)

In [394]:
# # ── Načítanie zo skutočného datasetu ──────────────────────────────────────
# import glob
#
# dataset_path = '../../../data/datasets/research_leafs_binary/'
# npy_files = sorted(glob.glob(dataset_path + '*.npy'))
# print(f"Dostupné súbory: {len(npy_files)}")
#
# if npy_files:
#     # Zobraziť prvý obrázok z datasetu
#     img_real = np.load(npy_files[0])
#     print(f"Načítaný: {npy_files[0]}")
#     print(f"Tvar: {img_real.shape}, Aktívnych px: {int(img_real.sum())}")
#     result_real = gbd_algorithm_complete(img_real, verbose=True)
#     print()
#     visualize_gbd_steps(result_real, img_real, cell_size=50)